# 01 - new domains, download and convert once (CPU runtime)

Glacial lakes, snow, forest and alpine glaciers, one public Sentinel-1 dataset
each. Every dataset is downloaded from its official source (for glacial lakes a Hugging
Face copy, checked against it), listed as it is on
disk (`inventory`), converted ONCE into our compact format (`prepared.py`,
SAR values, labels, one row of metadata per chip) and copied to Drive as a few
large files (`MyDrive/sar-transfer-part2/domains/`). Notebook 02 then runs the frozen
encoders on them.

**Runtime** - CPU with High-RAM is enough; no GPU. Check the free-disk line in
cell 3, since each raw archive lives only on this runtime's disk and is deleted with
it. Run the dataset cells one at a time; a cell that already finished skips its
work (downloads resume, a converted dataset on Drive is not redone).

| cell | dataset | download (exact, from the official listings) | peak runtime disk |
|---|---|---|---|
| 4 | Glacial-Lake-Bench (Hugging Face copy, checked file by file against the official release) + challenge set (glacial lakes) | 46.9 GB | about 101 GB (archive + extracted files) |
| 5 | ONERA SnowSAR (snow, French Alps) | 43.0 GB | about 75 GB (raw + up to 32 GB prepared) |
| 6 | ForTy v1, 32 train / 32 val / 32 test shards, worldwide (forest types) | about 34.4 GB | about 35 GB |
| 7 | GlaViTU, Alps tiles only (mountain glaciers) | 18.9 GB | about 20 GB |

Cells marked **YOUR CHOICE** must be set before that dataset is converted (all set, 2026-09-27).

Cell 8 uploads the CONVERTED datasets (not the raw downloads) to private Hugging Face repos
`<account>/sar-transfer-<dataset>` (the account is set in the cell; '-' for '_', i.e. glacial-lakes, snow, forty-v1, glavitu-alp). It needs `HF_WRITE_TOKEN` in the `.env` on Drive (`MyDrive/sar-transfer/.env`, shared with part one).


In [1]:
# --- cell - mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [7]:
# --- cell - paths, disk, RAM, helper ---
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path
import psutil
from sartransfer.domains import prepared as P

DRIVE = Path("/content/drive/MyDrive/sar-transfer-part2")  # part two; .env stays in MyDrive/sar-transfer
RAW_ROOT = Path("/content/domains_raw")            # raw archives: this runtime only
LOCAL_PREP = Path("/content/domains_prepared")     # converted here first (fast local disk)
DRIVE_PREP = DRIVE / "domains"                     # then copied to Drive: a few large files per dataset
for p in (RAW_ROOT, LOCAL_PREP, DRIVE_PREP):
    p.mkdir(parents=True, exist_ok=True)
print(f"CPU cores {os.cpu_count()} | RAM {psutil.virtual_memory().total / 1e9:.0f} GB | "
      f"free disk {shutil.disk_usage('/content').free / 1e9:.0f} GB")


def prepare(M, keep_raw=False, **opts):
    """download -> inventory -> convert -> validate -> copy to Drive, for one converter module M."""
    if not P.check_complete(DRIVE_PREP / M.DATASET):       # [] = a complete copy on Drive
        print(f"{M.DATASET}: already prepared on Drive -> skip"); P.validate(DRIVE_PREP, M.DATASET); return
    missing = [pip for mod, pip in getattr(M, "REQUIRES", []) if importlib.util.find_spec(mod) is None]
    if missing:
        print("installing", missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    raw = RAW_ROOT / M.DATASET
    t0 = time.perf_counter()
    M.download(raw, **opts.get("download", {}))
    print(f"download {(time.perf_counter() - t0) / 60:.1f} min | free disk {shutil.disk_usage('/content').free / 1e9:.0f} GB")
    M.inventory(raw, **opts.get("inventory", {}))
    t0 = time.perf_counter()
    M.convert(raw, LOCAL_PREP, **opts.get("convert", {}))
    print(f"convert {(time.perf_counter() - t0) / 60:.1f} min")
    P.validate(LOCAL_PREP, M.DATASET)
    t0 = time.perf_counter()
    tmp, final = DRIVE_PREP / f".{M.DATASET}.partial", DRIVE_PREP / M.DATASET
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(LOCAL_PREP / M.DATASET, tmp, ignore=shutil.ignore_patterns(".*"))   # never copy leftover part files
    bad = P.check_complete(tmp)
    if bad:
        raise IOError(f"{M.DATASET}: copy to Drive incomplete: {bad}")
    shutil.rmtree(final, ignore_errors=True)                  # an older, incomplete copy
    tmp.rename(final)                                         # only a complete copy gets the final name
    print(f"copied to Drive in {(time.perf_counter() - t0) / 60:.1f} min: {final}")
    if not keep_raw:
        shutil.rmtree(raw, ignore_errors=True)
        print(f"raw files removed from this runtime | free disk {shutil.disk_usage('/content').free / 1e9:.0f} GB")

CPU cores 8 | RAM 55 GB | free disk 164 GB


In [8]:
# --- cell - GLACIAL LAKES -- Glacial-Lake-Bench + challenge set, 46.9 GB download ---
from sartransfer.domains import glacial_lakes as M
prepare(M, download={
    "bench_source": "hf",        # GLB from its Hugging Face copy (Sk-21/Cryo-Bench, 44.3 GB .tar.gz): sha256 checked,
                                 # extracted (56.4 GB; peak disk about 101 GB), then every image and mask compared
                                 # with the Zenodo release (size + CRC-32); the challenge set comes from Zenodo.
                                 # "zenodo" = the official zip instead, read in place (peak disk about 53 GB)
}, convert={
    "holdout_regions": ("SEE",), # YOUR CHOICE: ("SEE",) also holds South Asia East out as test_region_SEE;
                                 # () = official random split only (93 % of its test chips share a
                                 # Sentinel-2 scene with train)
    "include_challenge": True,   # the hard-case challenge set becomes split test_challenge
})

[skip] /content/domains_raw/glacial_lakes/GLB: extraction complete
[zenodo] central directory: 38,240 files (2 range requests, 5.6 MB, 6 s); .tif per folder {'ann_dir/test': 1907, 'ann_dir/train': 15300, 'ann_dir/val': 1912, 'img_dir/test': 1907, 'img_dir/train': 15300, 'img_dir/val': 1912}
[crc32] 38,238 files of GLB/, 8 threads ...
[note] 12 files of the Hugging Face copy are not in the Zenodo release and are left out, e.g. GLB/test/masks/ACS_S2A_20WMT_20200821_1_L2A_82.tif, GLB/test/masks/ACS_S2B_18WVE_20200725_1_L2A_185.tif, GLB/test/masks/AC_S2A_16XDK_20200809_2_L2A_4.tif, GLB/test/masks/AC_S2A_19XDL_20200808_2_L2A_175.tif
GLB from Hugging Face: 19,119 of 19,119 pairs byte-identical to the Zenodo release (size + CRC-32 of every image and mask; train 15,300, val 1,912, test 1,907; 4.9 min); left out 6 extra image/mask pairs of the Hugging Face copy that Zenodo does not have
  Glacial-Lake-Challenge.zip: 0.01 / 2.59 GB, 0.9 MB/s avg
  Glacial-Lake-Challenge.zip: 0.05 / 2.59 GB, 1.1 

In [12]:
# --- cell - SNOW -- ONERA SnowSAR, 43.0 GB download (no choices needed) ---
from sartransfer.domains import snow as M
prepare(M)                       # train and score on the observed MODIS labels, 512 px chips

README.md: complete and md5-verified earlier, skipped
snowsar_dataset.zip: complete and md5-verified earlier, skipped
download 0.0 min | free disk 164 GB
/content/domains_raw/snow/snowsar_dataset.zip: 43,036,387,541 bytes (published 43,036,387,541); README.md present; md5 verified by download()
981 zip entries; top-level folders: ['projection', 'reference_images', 'stats', 'test', 'train', 'validation']

train -> train (Guil): 88 scenes, orbits {66: 32, 88: 27, 139: 29}, dates 2018-09-06..2019-05-29, labels ['raw', 'cni', 'ks']
   {'split': 'train', 'file': '139_20180906.tif', 'shape': (1872, 8889), 'label_shape': (1872, 8889), 'table1_shape': (1872, 8889), 'shape_vs_table1': 'same', 'sar_bands': 2, 'sar_dtypes': ('float32', 'float32'), 'sar_nodata_tag': (None, None), 'descriptions': (None, None), 'crs': None, 'sar_compress': None, 'label_dtype': 'float32', 'label_nodata_tag': None, 'label_compress': 'DEFLATE', 'nan_share': 0.0, 'inf_pixels': 0, 'out_of_f16': 0, 'zero_pairs': 0, 'zero_

In [13]:
# --- cell - FOREST -- ForTy v1 shard subset, worldwide, about 34.4 GB download ---
from sartransfer.domains import forest as M
SHARDS = {"train": 32, "validation": 32, "test": 32} # DECIDED 2026-09-27: worldwide, 32/32/32 shards
                                                     # (val/test 618/636 chips; not the full 1.1 TB)
ORBIT = "desc"                                       # DECIDED 2026-09-27: descending (84 % of train pixel-steps valid; asc 69 %)
prepare(M, download={"shards": SHARDS}, inventory={"orbit": ORBIT}, convert={
    "shards": SHARDS,
    "orbit": ORBIT,                                  # the inventory above checks the same orbit
})

# Europe count: how many chips per split fall in
# Europe and in Germany + neighbours, from each chip's lat/lon. Rough boxes, fixed here:
# Europe lat 35..72, lon -25..45; Germany + neighbours lat 45.5..55.5, lon 2..19.
import pandas as pd
for split in P.load_manifest(DRIVE_PREP, M.DATASET)["splits"]:
    m = P.read_meta(DRIVE_PREP / M.DATASET / f"{split}_meta.csv")
    lat, lon = pd.to_numeric(m["lat"]), pd.to_numeric(m["lon"])
    eu = lat.between(35, 72) & lon.between(-25, 45)
    de = lat.between(45.5, 55.5) & lon.between(2, 19)
    print(f"{split:>11}: {len(m):6,} chips | Europe {int(eu.sum()):5,} ({eu.mean():.1%}) | "
          f"Germany + neighbours {int(de.sum()):5,} ({de.mean():.1%})")

ForTy v1: 98 files, 34.41 GB -> /content/domains_raw/forty_v1
  [1/98] ok   features.json  0.00/34.41 GB  0.0 min
  [2/98] ok   dataset_info.json  0.00/34.41 GB  0.0 min
  [3/98] ok   forty_v1-train.tfrecord-00000-of-01024  0.76/34.41 GB  0.1 min
  [4/98] ok   forty_v1-train.tfrecord-00001-of-01024  1.64/34.41 GB  0.3 min
  [5/98] ok   forty_v1-train.tfrecord-00002-of-01024  2.44/34.41 GB  0.4 min
  [6/98] ok   forty_v1-train.tfrecord-00003-of-01024  3.26/34.41 GB  0.5 min
  [7/98] ok   forty_v1-train.tfrecord-00004-of-01024  4.11/34.41 GB  0.7 min
  [8/98] ok   forty_v1-train.tfrecord-00005-of-01024  5.07/34.41 GB  0.8 min
  [9/98] ok   forty_v1-train.tfrecord-00006-of-01024  5.91/34.41 GB  0.9 min
  [10/98] ok   forty_v1-train.tfrecord-00007-of-01024  6.77/34.41 GB  1.0 min
  [11/98] ok   forty_v1-train.tfrecord-00008-of-01024  7.66/34.41 GB  1.1 min
  [12/98] ok   forty_v1-train.tfrecord-00009-of-01024  8.53/34.41 GB  1.3 min
  [13/98] ok   forty_v1-train.tfrecord-00010-of-01024  9.

In [14]:
# --- cell - ALPINE GLACIERS -- GlaViTU Alps tiles, 18.9 GB download ---
from sartransfer.domains import alpine as M
prepare(M, download={"subregions": ("ALP",)}, convert={
    "subregions": ("ALP",),
    "orbit_rule": "coverage",    # per tile, the orbit with more valid co-pol pixels (ascending on a tie)
})

20230905_train_global_ps384.hdf5: 177 ALP tiles, 0 already local, fetching 177 (11.44 GB); metadata took 1545 requests, 50.6 MB
  1/177 ALP-10-6: 0.08 / 11.44 GB, 6.4 MB/s
  2/177 ALP-11-7: 0.17 / 11.44 GB, 12.7 MB/s
  3/177 ALP-11-8: 0.25 / 11.44 GB, 19.0 MB/s
  4/177 ALP-12-3: 0.34 / 11.44 GB, 25.0 MB/s
  5/177 ALP-12-5: 0.42 / 11.44 GB, 22.0 MB/s
  6/177 ALP-12-7: 0.51 / 11.44 GB, 23.1 MB/s
  7/177 ALP-13-4: 0.59 / 11.44 GB, 25.0 MB/s
  8/177 ALP-13-6: 0.68 / 11.44 GB, 27.3 MB/s
  9/177 ALP-14-12: 0.74 / 11.44 GB, 29.1 MB/s
  10/177 ALP-14-13: 0.81 / 11.44 GB, 27.7 MB/s
  11/177 ALP-14-14: 0.87 / 11.44 GB, 28.4 MB/s
  12/177 ALP-14-3: 0.93 / 11.44 GB, 27.3 MB/s
  13/177 ALP-14-4: 1.00 / 11.44 GB, 29.1 MB/s
  14/177 ALP-14-5: 1.06 / 11.44 GB, 28.4 MB/s
  15/177 ALP-14-9: 1.13 / 11.44 GB, 29.7 MB/s
  16/177 ALP-15-11: 1.19 / 11.44 GB, 28.1 MB/s
  17/177 ALP-15-12: 1.25 / 11.44 GB, 29.4 MB/s
  18/177 ALP-15-13: 1.32 / 11.44 GB, 28.5 MB/s
  19/177 ALP-15-4: 1.38 / 11.44 GB, 29.8 MB/s
  

In [ ]:
# --- cell - UPLOAD the converted datasets to private Hugging Face repos (<account>/sar-transfer-<dataset>, with '-' for '_', i.e. glacial-lakes, snow, forty-v1, glavitu-alp) ---
# Converted files only (decided 2026-09-27). Each repo is created PRIVATE; the upload refuses a
# public repo or an incomplete folder and checks every file's size afterwards. The write token is
# read from .env for this cell only (env.secret), never shown and never put into os.environ.
from sartransfer.env import secret
from sartransfer.domains import hub

HF_OWNER = "saverin00"                               # your Hugging Face account for the private copies
token = secret("HF_WRITE_TOKEN", "upload to your private Hugging Face repos")
for name in sorted(d.name for d in DRIVE_PREP.iterdir() if not d.name.startswith(".") and (d / "manifest.json").exists()):
    src = LOCAL_PREP if P.check_complete(LOCAL_PREP / name) == [] else DRIVE_PREP   # local disk is faster
    t0 = time.perf_counter()
    try:
        hub.upload(src, name, HF_OWNER, token=token)
    except Exception as e:
        print(f"FAILED {name}: {type(e).__name__}: {str(e)[:300]}")
    print(f"{name}: {(time.perf_counter() - t0) / 60:.1f} min\n")
del token